# GeoSuitability AI - Dashboard Launcher

This notebook starts the BI dashboard (`app.py`, Streamlit) as a local background process, straight
from PyCharm - no terminal needed, no Docker.

1. Run the first code cell below - a `http://localhost:<port>` link will appear; open it in your browser.
2. Use the dashboard in the browser as usual.
3. When you're done, run the last cell to shut the server down.

In [1]:
import atexit
import os
import socket
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

from IPython.display import Markdown, display

# self-correcting cwd, same convention as scripts/preprocessing/*.ipynb
NOTEBOOK_DIR = Path(os.path.abspath("."))
if NOTEBOOK_DIR.name != "dashboard":
    NOTEBOOK_DIR = NOTEBOOK_DIR / "scripts" / "dashboard"
os.chdir(NOTEBOOK_DIR)


def find_free_port(start=8501, tries=20):
    for port in range(start, start + tries):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            if s.connect_ex(("127.0.0.1", port)) != 0:
                return port
    raise RuntimeError("No free port found.")


PORT = find_free_port()

streamlit_proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app.py",
     "--server.headless", "true", "--server.port", str(PORT)],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)


def _stop_dashboard():
    if streamlit_proc.poll() is None:
        streamlit_proc.terminate()


atexit.register(_stop_dashboard)  # safety net if the kernel dies without running the stop cell below

url = f"http://localhost:{PORT}"
for _ in range(30):
    try:
        urllib.request.urlopen(url, timeout=1)
        break
    except Exception:
        time.sleep(1)
else:
    print("Server did not respond within 30 seconds - check the environment for errors.")

display(Markdown(
    f"**Dashboard is running at:** [{url}]({url})\n\n"
    "Open the link above in your browser. When you're done, run the last cell to shut the server down."
))

**Dashboard is running at:** [http://localhost:8501](http://localhost:8501)

Open the link above in your browser. When you're done, run the last cell to shut the server down.

---

### Done using the dashboard?

Run the cell below to shut down the local server.

In [2]:
streamlit_proc.terminate()
try:
    streamlit_proc.wait(timeout=10)
except subprocess.TimeoutExpired:
    streamlit_proc.kill()
print("Dashboard stopped.")

Dashboard stopped.
